# Tutorial de Queries — COMEXSTAT (DuckDB + Pandas)

Este notebook demonstra como explorar e consultar os dados de comércio exterior brasileiro
armazenados no DuckDB, usando pandas para visualização.

> **Importante:** Os dados são muito grandes para exibir por inteiro. Todas as queries
> usam `LIMIT`, agregações ou filtragens para manter a saída concisa.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

conn = duckdb.connect('./db/comexstat.duckdb', read_only=True)
print('Conexao aberta com sucesso.')

---
## 1. Explorando o Schema

In [ ]:
tables = conn.execute("SHOW TABLES").df()
tables

In [ ]:
conn.execute("DESCRIBE exp").df()
# repita para outras tabelas: DESCRIBE imp, DESCRIBE ncm, DESCRIBE pais, DESCRIBE via, etc.

In [ ]:
conn.execute("""
    SELECT
        'exp' AS tabela, COUNT(*) AS registros
    FROM exp
    UNION ALL
    SELECT
        'imp' AS tabela, COUNT(*) AS registros
    FROM imp
""").df()

---
## 2. Amostragem com LIMIT

Sempre que quiser inspecionar linhas específicas, use `LIMIT`.

In [ ]:
conn.execute("SELECT * FROM exp LIMIT 5").df()

In [ ]:
conn.execute("SELECT CO_NCM, NO_NCM_POR FROM ncm LIMIT 5").df()

---
## 3. Agregações Básicas

Para dados grandes, sempre agregue antes de exibir.

In [ ]:
conn.execute("""
    SELECT
        'Exportação' AS tipo,
        SUM(VL_FOB) AS fob_total,
        SUM(KG_LIQUIDO) AS kg_total,
        COUNT(*) AS qtd_registros
    FROM exp
    UNION ALL
    SELECT
        'Importação' AS tipo,
        SUM(VL_FOB) AS fob_total,
        SUM(KG_LIQUIDO) AS kg_total,
        COUNT(*) AS qtd_registros
    FROM imp
""").df()

---
## 4. Evolução Anual do Comércio

Exportações e importações agrupadas por ano.

In [ ]:
df_ano = conn.execute("""
    SELECT
        CO_ANO AS ano,
        'Exportação' AS tipo,
        SUM(VL_FOB) AS fob
    FROM exp
    GROUP BY CO_ANO
    UNION ALL
    SELECT
        CO_ANO AS ano,
        'Importação' AS tipo,
        SUM(VL_FOB) AS fob
    FROM imp
    GROUP BY CO_ANO
    ORDER BY ano, tipo
""").df()

df_ano

In [ ]:
pivot = df_ano.pivot(index='ano', columns='tipo', values='fob')
pivot.plot(kind='bar', figsize=(10, 5), ylabel='FOB (USD)', xlabel='Ano',
           title='Evolução Anual do Comércio Exterior')
plt.tight_layout()
plt.show()

---
## 5. Top 10 Estados por Exportação

In [ ]:
df_uf = conn.execute("""
    SELECT
        e.SG_UF_NCM AS uf,
        u.NO_UF AS estado,
        u.NO_REGIAO AS regiao,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN uf u ON e.SG_UF_NCM = u.SG_UF
    GROUP BY e.SG_UF_NCM, u.NO_UF, u.NO_REGIAO
    ORDER BY fob_total DESC
    LIMIT 10
""").df()

df_uf

In [ ]:
df_uf.set_index('uf')['fob_total'].plot(
    kind='barh', figsize=(8, 5),
    xlabel='FOB (USD)', title='Exportações por Estado'
)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 6. Top 10 Produtos (NCM) por Exportação

Juntando a tabela `exp` com a tabela de referência `ncm` para obter o nome do produto.

In [ ]:
df_ncm = conn.execute("""
    SELECT
        e.CO_NCM AS ncm,
        n.NO_NCM_POR AS produto,
        SUM(e.VL_FOB) AS fob_total,
        SUM(e.KG_LIQUIDO) AS kg_total
    FROM exp e
    LEFT JOIN ncm n ON e.CO_NCM = n.CO_NCM
    GROUP BY e.CO_NCM, n.NO_NCM_POR
    ORDER BY fob_total DESC
    LIMIT 10
""").df()

df_ncm

In [ ]:
df_ncm.set_index('ncm')['fob_total'].plot(
    kind='barh', figsize=(8, 5),
    xlabel='FOB (USD)', title='Produtos Mais Exportados'
)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 7. Top 10 Parceiros Comerciais

In [ ]:
df_pais = conn.execute("""
    SELECT
        e.CO_PAIS AS cod_pais,
        p.NO_PAIS AS pais,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN pais p ON e.CO_PAIS = p.CO_PAIS
    GROUP BY e.CO_PAIS, p.NO_PAIS
    ORDER BY fob_total DESC
    LIMIT 10
""").df()

df_pais

In [ ]:
df_pais.set_index('pais')['fob_total'].plot(
    kind='bar', figsize=(10, 5),
    ylabel='FOB (USD)', title='Países Mais Exportadores'
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Série Mensal (ano filtrado)

Para ver a sazonalidade, filtrar por um ano específico.

In [ ]:
ANO = 2025

df_mes = conn.execute("""
    SELECT
        CO_MES AS mes,
        'Exportação' AS tipo,
        SUM(VL_FOB) AS fob
    FROM exp
    WHERE CO_ANO = ?
    GROUP BY CO_MES
    UNION ALL
    SELECT
        CO_MES AS mes,
        'Importação' AS tipo,
        SUM(VL_FOB) AS fob
    FROM imp
    WHERE CO_ANO = ?
    GROUP BY CO_MES
    ORDER BY mes, tipo
""", [ANO, ANO]).df()

df_mes

In [ ]:
pivot_mes = df_mes.pivot(index='mes', columns='tipo', values='fob')
pivot_mes.plot(kind='line', marker='o', figsize=(10, 5),
               ylabel='FOB (USD)', xlabel='Mês',
               title='Comércio Exterior Mensal')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Comércio por Via de Transporte

In [ ]:
df_via = conn.execute("""
    SELECT
        e.CO_VIA AS cod_via,
        v.NO_VIA AS via,
        'Exportação' AS tipo,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN via v ON e.CO_VIA = v.CO_VIA
    GROUP BY e.CO_VIA, v.NO_VIA
    UNION ALL
    SELECT
        e.CO_VIA AS cod_via,
        v.NO_VIA AS via,
        'Importação' AS tipo,
        SUM(e.VL_FOB) AS fob_total
    FROM imp e
    LEFT JOIN via v ON e.CO_VIA = v.CO_VIA
    GROUP BY e.CO_VIA, v.NO_VIA
    ORDER BY via, tipo
""").df()

df_via

In [ ]:
pivot_via = df_via.pivot_table(index='via', columns='tipo', values='fob_total', aggfunc='sum')
pivot_via.plot(kind='bar', figsize=(10, 5), ylabel='FOB (USD)',
               title='Exportações e Importações por Via de Transporte')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 10. Balança Comercial por Ano

In [ ]:
df_bal = conn.execute("""
    WITH
    exp_agg AS (
        SELECT CO_ANO AS ano, SUM(VL_FOB) AS fob_exp
        FROM exp GROUP BY CO_ANO
    ),
    imp_agg AS (
        SELECT CO_ANO AS ano, SUM(VL_FOB) AS fob_imp
        FROM imp GROUP BY CO_ANO
    )
    SELECT
        COALESCE(e.ano, i.ano) AS ano,
        e.fob_exp,
        i.fob_imp,
        e.fob_exp - i.fob_imp AS balanca
    FROM exp_agg e
    FULL OUTER JOIN imp_agg i ON e.ano = i.ano
    ORDER BY ano
""").df()

df_bal

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in df_bal['balanca']]
ax.bar(df_bal['ano'].astype(str), df_bal['balanca'], color=colors)
ax.set_ylabel('Balança (USD)')
ax.set_xlabel('Ano')
ax.set_title('Balança Comercial (Exportação − Importação)')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

---
## 11. Agregação por Capítulo SH (6 dígitos)

A tabela `ncm` já contém o código `CO_SH6`. Para obter o nome da seção, fazemos JOIN com `ncm_sh`.

In [ ]:
df_sh = conn.execute("""
    SELECT
        n.CO_SH6 AS sh6,
        s.NO_SEC_POR AS secao,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN ncm n ON e.CO_NCM = n.CO_NCM
    LEFT JOIN ncm_sh s ON n.CO_SH6 = s.CO_SH6
    GROUP BY n.CO_SH6, s.NO_SEC_POR
    ORDER BY fob_total DESC
    LIMIT 15
""").df()

df_sh

In [ ]:
df_sh.set_index('sh6')['fob_total'].plot(
    kind='bar', figsize=(12, 5),
    ylabel='FOB (USD)', title='Capítulos SH Mais Exportados'
)
plt.tight_layout()
plt.show()

---
## 12. Preço Médio por Produto (FOB/KG)

Calcula o preço médio por quilo para os 15 maiores produtos exportados.

In [ ]:
df_preco = conn.execute("""
    SELECT
        e.CO_NCM AS ncm,
        n.NO_NCM_POR AS produto,
        SUM(e.VL_FOB) AS fob_total,
        SUM(e.KG_LIQUIDO) AS kg_total,
        SUM(e.VL_FOB) / NULLIF(SUM(e.KG_LIQUIDO), 0) AS preco_medio_kg
    FROM exp e
    LEFT JOIN ncm n ON e.CO_NCM = n.CO_NCM
    GROUP BY e.CO_NCM, n.NO_NCM_POR
    HAVING SUM(e.KG_LIQUIDO) > 0
    ORDER BY fob_total DESC
    LIMIT 15
""").df()

df_preco

---
## 13. Top 5 Produtos por Estado (Spot check)

Exemplo: quais são os principais produtos exportados por SP.

In [ ]:
UF = 'SP'

df_uf_prod = conn.execute("""
    SELECT
        e.CO_NCM AS ncm,
        n.NO_NCM_POR AS produto,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN ncm n ON e.CO_NCM = n.CO_NCM
    WHERE e.SG_UF_NCM = ?
    GROUP BY e.CO_NCM, n.NO_NCM_POR
    ORDER BY fob_total DESC
    LIMIT 5
""", [UF]).df()

df_uf_prod

---
## 14. Explorando as Tabelas de Referência

Código de país, UF, via de transporte, URF, blocos comerciais.

In [ ]:
conn.execute("SELECT * FROM pais LIMIT 5").df()

In [ ]:
conn.execute("SELECT * FROM via ORDER BY CO_VIA").df()

In [ ]:
conn.execute("SELECT * FROM uf").df()

In [ ]:
conn.execute("""
    SELECT DISTINCT NO_BLOCO
    FROM pais_bloco
    ORDER BY NO_BLOCO
""").df()

---
## 15. JOIN Multi-tabela: Exportações por País + Via + Ano

Juntando `exp`, `pais`, `via` em uma única query.

In [ ]:
df_multi = conn.execute("""
    SELECT
        e.CO_ANO AS ano,
        p.NO_PAIS AS pais,
        v.NO_VIA AS via,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN pais p ON e.CO_PAIS = p.CO_PAIS
    LEFT JOIN via v ON e.CO_VIA = v.CO_VIA
    WHERE e.CO_ANO = 2025
    GROUP BY e.CO_ANO, p.NO_PAIS, v.NO_VIA
    ORDER BY fob_total DESC
    LIMIT 10
""").df()

df_multi

---
## 16. Exportações por Bloco Econômico

In [ ]:
df_bloco = conn.execute("""
    SELECT
        b.NO_BLOCO AS bloco,
        SUM(e.VL_FOB) AS fob_total
    FROM exp e
    LEFT JOIN pais_bloco b ON e.CO_PAIS = b.CO_PAIS
    GROUP BY b.NO_BLOCO
    ORDER BY fob_total DESC
""").df()

df_bloco

In [ ]:
df_bloco.dropna().set_index('bloco')['fob_total'].plot(
    kind='barh', figsize=(8, 5),
    xlabel='FOB (USD)', title='Exportações por Bloco Econômico'
)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 17. Dica: Paginação com DuckDB

Para explorar registros sem carregar tudo na memória:

In [ ]:
PAGE_SIZE = 10
PAGE = 0  # mude para navegar

offset = PAGE * PAGE_SIZE
df_page = conn.execute("""
    SELECT CO_ANO, CO_MES, CO_NCM, SG_UF_NCM, VL_FOB
    FROM exp
    ORDER BY VL_FOB DESC
    LIMIT ? OFFSET ?
""", [PAGE_SIZE, offset]).df()

print(f'Página {PAGE} (offset {offset})')
df_page

---
## Fechando a Conexão

In [ ]:
conn.close()
print('Conexao fechada.')